<a href="https://colab.research.google.com/github/Jhnfdn/TRAINING-EMOTION-DATASET/blob/main/emotion_FER2013_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Dropout, Flatten, Dense, ReLU, DepthwiseConv2D, GlobalAveragePooling2D, Concatenate, Activation,
                                     BatchNormalization, Input, SeparableConv2D, Multiply, Reshape, Add)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt

In [2]:
import kagglehub
import os
import shutil
# Download latest version
path = kagglehub.dataset_download("msambare/fer2013")

# Define the destination directory
destination = "/content/emotions/fer2013"

# Ensure the destination directory exists
os.makedirs(destination, exist_ok=True)

# Move the downloaded files
shutil.move(path, destination)

print("Dataset moved to:", destination)

Using Colab cache for faster access to the 'fer2013' dataset.


OSError: [Errno 30] Read-only file system: '/kaggle/input/fer2013/train/happy'

In [ ]:
import os, shutil, zipfile, tf2onnx
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

# 1. UNZIP AND FUSE
print(" Extracting CK+ data...")
with zipfile.ZipFile("ck_data.zip", 'r') as z:
    z.extractall("ck_temp")

# We use the 'train' folder from the previous notebook as our master source
master_dir = 'train'

# Map your CK+ folders to project labels
mapping = {'happy': 'happy', 'contempt': 'neutral', 'sadness': 'sad', 'fear': 'stressed'}

print(" Merging datasets and renaming folders...")
for ck_folder, proj_label in mapping.items():
    src = f"ck_temp/{ck_folder}"
    dst = f"{master_dir}/{proj_label}" # We add CK+ into the FER folder
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        for img in os.listdir(src):
            shutil.copy(os.path.join(src, img), os.path.join(dst, f"ck_{img}"))

# Ensure the FER 'surprise' folder is also renamed to 'stressed'
if os.path.exists(f"{master_dir}/surprise"):
    os.rename(f"{master_dir}/surprise", f"{master_dir}/stressed")

print("Master Dataset Ready.")

# 2. DATA GENERATORS (The Splitting)
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    validation_split=0.2 # <--- 20% OF ALL DATA IS HELD BACK FOR TESTING
)

# Create Training Set (80%)
train_gen = datagen.flow_from_directory(
    master_dir, target_size=(64,64), batch_size=32,
    class_mode='categorical', color_mode='rgb', subset='training'
)

# Create Validation Set (20%)
val_gen = datagen.flow_from_directory(
    master_dir, target_size=(64,64), batch_size=32,
    class_mode='categorical', color_mode='rgb', subset='validation'
)

# 3. BUILD AND TRAIN (A* Strategy)
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(64,64,3))
base.trainable = True # Unfreeze for better accuracy
for layer in base.layers[:100]: layer.trainable = False # Keep early layers stable

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')
])

model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

print("\n Training A* Fusion Model...")
model.fit(train_gen, validation_data=val_gen, epochs=30)

# 4. EXPORT
print("\n Exporting to ONNX...")
spec = (tf.TensorSpec((None, 64, 64, 3), tf.float32, name="input"),)
tf2onnx.convert.from_keras(model, input_signature=spec, opset=13, output_path="emotion_model_trained.onnx")

print("\n ALL DONE! Download 'emotion_model_trained.onnx'")
print(f"Final Label Order: {list(train_gen.class_indices.keys())}")

In [ ]:
# Define the dataset directory
dataset_test = "/content/emotions/fer2013/1/test"
dataset_train = "/content/emotions/fer2013/1/train"

# Create an ImageDataGenerator instance
train_datagen = ImageDataGenerator(
    width_shift_range = 0.1,        # Randomly shift the width of images by up to 10%
    height_shift_range = 0.1,       # Randomly shift the height of images by up to 10%
    horizontal_flip = True,         # Flip images horizontally at random
    rescale = 1./255,               # Rescale pixel values to be between 0 and 1
    shear_range=0.1,
    zoom_range=0.1,
    rotation_range=10,
    validation_split = 0.2          # Set aside 20% of the data for validation
)

validation_datagen = ImageDataGenerator(
    rescale = 1./255,               # Rescale pixel values to be between 0 and 1
    validation_split = 0.2          # Set aside 20% of the data for validation
)

# Load training data
train_generator = train_datagen.flow_from_directory(
    dataset_train,
    target_size=(48, 48),  # Adjust based on your CNN input size
    batch_size=32,
    color_mode = 'grayscale',
    class_mode='categorical',  # Use 'sparse' for integer labels
    subset='training'
)

# Load validation data
val_generator = validation_datagen.flow_from_directory(
    dataset_test,
    target_size=(48, 48),
    batch_size=32,
    color_mode = 'grayscale',
    class_mode='categorical',
    subset='validation'
)

# Print class labels
print("Class labels:", train_generator.class_indices)

In [ ]:
def se_block(input_tensor, reduction=16):
    """Squeeze and Excitation block to recalibrate channel-wise feature responses."""
    channels = int(input_tensor.shape[-1])
    se = GlobalAveragePooling2D()(input_tensor)
    se = Dense(channels // reduction, activation='relu')(se)
    se = Dense(channels, activation='sigmoid')(se)
    se = Reshape((1, 1, channels))(se)
    return Multiply()([input_tensor, se])

def residual_se_block(x, filters, kernel_size=3, strides=1, dropout_rate=0.25, use_se=True):
    """Residual block with separable convolutions and optional SE block."""
    shortcut = x

    # First separable convolution
    x = SeparableConv2D(filters, kernel_size, padding='same', strides=strides, activation='relu')(x)
    x = BatchNormalization()(x)

    # Second separable convolution
    x = SeparableConv2D(filters, kernel_size, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)

    # Optionally apply squeeze and excitation
    if use_se:
        x = se_block(x, reduction=16)

    # Adjust the shortcut if necessary
    if strides != 1 or int(shortcut.shape[-1]) != filters:
        shortcut = Conv2D(filters, (1, 1), strides=strides, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add residual connection and activate
    x = Add()([x, shortcut])
    x = Activation('relu')(x)

    # Downsample spatially and add dropout for regularization
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(dropout_rate)(x)
    return x

def create_best_emotion_model(input_shape=(48, 48, 1), num_classes=7):
    inputs = Input(shape=input_shape)

    # Initial standard convolution layer to expand channels
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)

    # Residual-SE Block 1: filters=32
    x = residual_se_block(x, filters=32, kernel_size=3, strides=1, dropout_rate=0.25, use_se=True)

    # Residual-SE Block 2: filters=64
    x = residual_se_block(x, filters=64, kernel_size=3, strides=1, dropout_rate=0.25, use_se=True)

    # Residual-SE Block 3: filters=128
    x = residual_se_block(x, filters=128, kernel_size=3, strides=1, dropout_rate=0.25, use_se=True)

    # One more block with increased filters for deeper features
    x = residual_se_block(x, filters=256, kernel_size=3, strides=1, dropout_rate=0.3, use_se=True)

    # Global Average Pooling reduces parameters versus Flattening
    x = GlobalAveragePooling2D()(x)

    # Fully connected layers for final classification
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

# Create the model instance and display its summary.
model = create_best_emotion_model()
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Early stopping to prevent overfitting
early_stopping = EarlyStopping(
    monitor='val_loss',  # Stop when validation loss stops improving
    patience=50,  # Number of epochs to wait before stopping
    restore_best_weights=True  # Restore best weights
)

# Model checkpoint to save the best model
checkpoint = ModelCheckpoint(
    "emotions.h5",  # File name
    monitor='val_accuracy',  # Save the model with the best validation accuracy
    save_best_only=True,
    verbose=1
)

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    validation_data=val_generator,
    validation_steps=len(val_generator),
    epochs=300,  # Can be higher since early stopping prevents unnecessary training
    batch_size=32,
    verbose=1,
    callbacks=[early_stopping,checkpoint]  # Add callbacks here
)

In [ ]:
# Load the best saved model
best_model = tf.keras.models.load_model("emotions.h5")

# Evaluate performance
loss, accuracy = best_model.evaluate(val_generator)
print(f"Best Model Validation Accuracy: {accuracy * 100:.2f}%")

In [ ]:
plt.figure()
plt.title('Train and Validation Accuracy')
plt.plot(history.history['accuracy'], label='Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.show()

plt.figure()
plt.title('Train and Validation Loss')
plt.plot(history.history['loss'], label='Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.show()

In [ ]:
imgs, labels = next(val_generator)

fig, axes = plt.subplots(nrows=1, ncols=5, figsize=(10, 3))
for i in range(5):
    image = np.expand_dims(imgs[i], axis=0)
    predict = best_model.predict(image)[0].argmax()
    label = np.argmax(labels[i], axis=0)

    axes[i].imshow(imgs[i], cmap='gray')
    axes[i].set_title(f'Truth: {label}, Predict: {predict}')
    axes[i].axis('off')

plt.show()


In [ ]:
from google.colab import files

files.download("emotions.h5")

In [ ]:
model.save("last.h5")

files.download("last.h5")

In [ ]:
# 1. INSTALL TOOLS FIRST
!pip install tf2onnx

import os, shutil, zipfile
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
import tf2onnx

# 2. UNZIP AND FUSE
print("📦 Extracting CK+ data...")
if not os.path.exists("ck_data.zip"):
    raise Exception("❌ Could not find ck_data.zip in the sidebar. Please upload it!")

with zipfile.ZipFile("ck_data.zip", 'r') as z:
    z.extractall("ck_temp")

# We use the 'train' folder from the previous notebook as our master source
master_dir = 'train'

# Map your CK+ folders to project labels (Matching your screenshot)
mapping = {'happy': 'happy', 'contempt': 'neutral', 'sadness': 'sad', 'fear': 'stressed'}

print("🔄 Merging datasets...")
for ck_folder, proj_label in mapping.items():
    src = f"ck_temp/{ck_folder}"
    dst = f"{master_dir}/{proj_label}"
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        for img in os.listdir(src):
            shutil.copy(os.path.join(src, img), os.path.join(dst, f"ck_{img}"))

# Ensure FER 'surprise' is also renamed to 'stressed'
if os.path.exists(f"{master_dir}/surprise"):
    os.rename(f"{master_dir}/surprise", f"{master_dir}/stressed")

print("✅ Master Dataset Ready.")

# 3. DATA GENERATORS (80/20 Split)
datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, horizontal_flip=True, validation_split=0.2)

train_gen = datagen.flow_from_directory(master_dir, target_size=(64,64), batch_size=32, class_mode='categorical', color_mode='rgb', subset='training')
val_gen = datagen.flow_from_directory(master_dir, target_size=(64,64), batch_size=32, class_mode='categorical', color_mode='rgb', subset='validation')

# 4. BUILD MODEL (A* Two-Stage Fine-Tuning)
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(64,64,3))

# --- STAGE 1: Warmup (Frozen Base) ---
base.trainable = False
model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')
])

print("\n🚀 Stage 1: Warming up the top layers...")
model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_gen, validation_data=val_gen, epochs=5)

# --- STAGE 2: True Fine-Tuning (Unfreeze partial base) ---
print("\n🚀 Stage 2: Fine-Tuning the brain...")
base.trainable = True
# Freeze first 100 layers to keep early features stable
for layer in base.layers[:100]: layer.trainable = False

# Use a MUCH slower learning rate for fine-tuning (Essential for A*)
model.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_gen, validation_data=val_gen, epochs=25)

# 5. EXPORT
print("\n📦 Exporting to ONNX...")
spec = (tf.TensorSpec((None, 64, 64, 3), tf.float32, name="input"),)
tf2onnx.convert.from_keras(model, input_signature=spec, opset=13, output_path="emotion_model_trained.onnx")

print("\n🎉 DONE! Download 'emotion_model_trained.onnx'")
print(f"Final Label Order: {list(train_gen.class_indices.keys())}")

In [ ]:
# 1. Install Tools
!pip install -q tf2onnx

import os, shutil, zipfile
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

# 2. DOWNLOAD FULL FER2013 (Verified stable repo with 28,000 images)
if not os.path.exists("fer2013_full"):
    print("📥 Downloading FULL FER2013 dataset...")
    !git clone https://github.com/Manish-Sahu21/Face-Emotion-Recognition-Using-CNN.git fer_repo
    !mv fer_repo/images fer2013_full
    !rm -rf fer_repo

# 3. FUSE WITH YOUR CK+ ZIP
if os.path.exists("ck_data.zip"):
    with zipfile.ZipFile("ck_data.zip", 'r') as z:
        z.extractall("ck_temp")
else:
    print(" Warning: ck_data.zip not found. Upload it for best results!")

# Create a clean Master Dataset folder
master_dir = 'A_STAR_DATASET'
emotions = ['happy', 'neutral', 'sad', 'stressed']
for emo in emotions: os.makedirs(f"{master_dir}/{emo}", exist_ok=True)

# Merge Full FER (Mapping fear to stressed)
fer_map = {'happy':'happy', 'neutral':'neutral', 'sad':'sad', 'fear':'stressed'}
print(" Merging 28,000 FER images...")
for src_emo, dst_emo in fer_map.items():
    src = f"fer2013_full/train/{src_emo}"
    if os.path.exists(src):
        for img in os.listdir(src):
            shutil.copy(os.path.join(src, img), f"{master_dir}/{dst_emo}/fer_{img}")

# Merge CK+
ck_map = {'happy':'happy', 'contempt':'neutral', 'sadness':'sad', 'fear':'stressed'}
print(" Merging CK+ images...")
if os.path.exists("ck_temp"):
    for src_emo, dst_emo in ck_map.items():
        src = f"ck_temp/{src_emo}"
        if os.path.exists(src):
            for img in os.listdir(src):
                shutil.copy(os.path.join(src, img), f"{master_dir}/{dst_emo}/ck_{img}")

print(f"FINAL DATASET READY: {sum([len(os.listdir(f'{master_dir}/{e}')) for e in emotions])} images found!")

In [ ]:
import os, shutil, zipfile

# 1. EXTRACT CK+ ZIP
print("Extracting CK+...")
if os.path.exists("ck_data.zip"):
    with zipfile.ZipFile("ck_data.zip", 'r') as z:
        z.extractall("ck_temp")
    print("CK+ Extracted.")
else:
    raise Exception("ERROR: Please upload ck_data.zip to the sidebar first!")

# 2. SETUP MASTER DIRECTORY
master_dir = 'merged_dataset'
emotions = ['happy', 'neutral', 'sad', 'stressed']
for emo in emotions:
    if os.path.exists(f"{master_dir}/{emo}"): shutil.rmtree(f"{master_dir}/{emo}")
    os.makedirs(f"{master_dir}/{emo}", exist_ok=True)

# 3. MERGE FER2013 (From the folder in your screenshot)
# Path from your image: emotions/fer2013/fer2013/train
fer_base = 'emotions/fer2013/fer2013/train'
fer_map = {'happy':'happy', 'neutral':'neutral', 'sad':'sad', 'fear':'stressed'}

print(" Fusing FER2013 images...")
for src_emo, dst_emo in fer_map.items():
    src = f"{fer_base}/{src_emo}"
    if os.path.exists(src):
        files = os.listdir(src)
        print(f"   Moving {len(files)} {src_emo} images...")
        for img in files:
            shutil.copy(os.path.join(src, img), f"{master_dir}/{dst_emo}/fer_{img}")

# 4. MERGE CK+
# Path: ck_temp/CK+48 (or similar based on your zip structure)
ck_map = {'happy':'happy', 'contempt':'neutral', 'sadness':'sad', 'fear':'stressed'}
# Search for the CK folder inside the temp
ck_search_root = 'ck_temp'
for root, dirs, files in os.walk('ck_temp'):
    if 'happy' in dirs:
        ck_search_root = root
        break

print(f"🔄 Fusing CK+ images from {ck_search_root}...")
for src_emo, dst_emo in ck_map.items():
    src = f"{ck_search_root}/{src_emo}"
    if os.path.exists(src):
        files = os.listdir(src)
        for img in files:
            shutil.copy(os.path.join(src, img), f"{master_dir}/{dst_emo}/ck_{img}")

# 5. FINAL COUNT
total = sum([len(os.listdir(f"{master_dir}/{e}")) for e in emotions])
print(f"\n SUCCESS! Your A* Dataset has {total} images.")

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import os

# 1. SETUP GENERATORS (Pointing to your renamed folder)
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    'merged_dataset',
    target_size=(64,64),
    batch_size=64,
    class_mode='categorical',
    color_mode='rgb',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    'merged_dataset',
    target_size=(64,64),
    batch_size=64,
    class_mode='categorical',
    color_mode='rgb',
    subset='validation'
)

# 2. BUILD THE MODEL (A* Fine-Tuning Strategy)
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(64,64,3))
base.trainable = True
# We unfreeze the last 40 layers of the brain for higher "intelligence"
for layer in base.layers[:-40]:
    layer.trainable = False

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')
])

# This reduces the learning rate when accuracy plateaus
lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7, verbose=1
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=7, restore_best_weights=True
)

model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# 4. START TRAINING
print("\n Starting Training (21,527 images)...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=40,
    callbacks=[lr_reducer, early_stop]
)

# 5. VISUALIZE FOR YOUR REPORT
plt.figure(figsize=(10, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title(' Multi-Dataset Emotion Model Accuracy')
plt.legend()
plt.show()

# 6. STABLE ONNX EXPORT
print("\n Exporting to ONNX...")
model.save("final_model.h5")
!pip install -q tf2onnx
!python -m tf2onnx.convert --keras final_model.h5 --output emotion_model_trained.onnx --opset 13

print("\n DONE! Download emotion_model_trained.onnx")
print(f"Final Label Order: {list(train_gen.class_indices.keys())}")

In [ ]:
print(" Stage 3: Aggressive Fine-Tuning")

# 1. Unfreeze the ENTIRE MobileNetV2 base
base.trainable = True


# using 1e-5 (0.00001)
model.compile(optimizer=Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# 3. Train for another 20-30 epochs
history_final = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    callbacks=[lr_reducer, early_stop]
)

# 4. EXPORT
print("\n Exporting the Stage 3 High-Accuracy model...")
model.save("a_star_model.h5")
!python -m tf2onnx.convert --keras a_star_model.h5 --output emotion_model_trained.onnx --opset 13

print("\n DONE! Download the new 'emotion_model_trained.onnx'")

In [ ]:
from sklearn.utils import class_weight
import numpy as np

# Calculate weights to balance the dataset
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
weight_dict = dict(enumerate(weights))

print(f"Applying Class Weights: {weight_dict}")

# Train with weights
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_gen, validation_data=val_gen, epochs=20, class_weight=weight_dict, callbacks=[lr_reducer, early_stop])

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Create a non-shuffled generator to get accurate results
eval_gen = datagen.flow_from_directory(
    'merged_dataset',
    target_size=(64,64),
    batch_size=64,
    class_mode='categorical',
    color_mode='rgb',
    subset='validation',
    shuffle=False # CRITICAL
)

# 2. Get predictions
print(" Analyzing model performance...")
Y_pred = model.predict(eval_gen)
y_pred = np.argmax(Y_pred, axis=1)
y_true = eval_gen.classes

# 3. Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=list(eval_gen.class_indices.keys()),
            yticklabels=list(eval_gen.class_indices.keys()))
plt.title('Final Confusion Matrix: Multi-Dataset Emotion Model')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

# 4. Show the A* Metrics
print("\n Detailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=list(eval_gen.class_indices.keys())))

In [ ]:
# 1. Save the latest weighted model to H5 format
model.save("weighted_final_model.h5")

# 2. Convert the latest model to ONNX
print("📦 Converting the weighted model to ONNX...")
!pip install -q tf2onnx
!python -m tf2onnx.convert --keras weighted_final_model.h5 --output emotion_model_weighted.onnx --opset 13

print("\n🎉 SUCCESS! Now download 'emotion_model_weighted.onnx' from the sidebar.")

In [ ]:
import cv2
import os


# Format: (label, filename)
video_files = [
    ('happy', 'mantasha_happy.mp4'), ('happy', 'papa_happy.mp4'), ('happy', 'jamal_happy.mp4'),
    ('neutral', 'mantasha_neutral.mp4'), ('neutral', 'papa_neutral.mp4'), ('neutral', 'jamal_neutral.mp4'),
    ('sad', 'mantasha_sad.mp4'), ('sad', 'papa_sad.mp4'), ('sad', 'jamal_sad.mp4'),
    ('stressed', 'mantasha_stressed.mp4'), ('stressed', 'papa_stressed.mp4'), ('stressed', 'jamal_stressed.mp4')
]

# Create temporary folder for family images
os.makedirs('family_dataset', exist_ok=True)
for emo in ['happy', 'neutral', 'sad', 'stressed']:
    os.makedirs(f'family_dataset/{emo}', exist_ok=True)

#  Extract Frames
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
print(" Extracting high-quality faces from videos...")
for emotion, filename in video_files:
    if not os.path.exists(filename):
        print(f"Skipping {filename}, not found.")
        continue

    cap = cv2.VideoCapture(filename)
    saved_count = 0
    frame_idx = 0  # Separate counter for video frames

    while cap.isOpened() and saved_count < 150:
        ret, frame = cap.read()
        if not ret: break

        # Increment frame index every time a frame is read
        frame_idx += 1

        # Only attempt detection every 3rd frame to get variety
        if frame_idx % 3 == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.3, 5)

            for (x, y, w, h) in faces:
                # Add a 15% margin around the face for better AI context
                offset = int(w * 0.15)
                y1, y2 = max(0, y-offset), min(frame.shape[0], y+h+offset)
                x1, x2 = max(0, x-offset), min(frame.shape[1], x+w+offset)

                crop = frame[y1:y2, x1:x2]

                # Save the image using saved_count so the filenames are unique
                save_name = f'family_dataset/{emotion}/fam_{filename}_{saved_count}.jpg'
                cv2.imwrite(save_name, crop)

                saved_count += 1
                break # Only take one face per frame

    cap.release()
    print(f" Extracted {saved_count} images from {filename}")

In [ ]:
import cv2
import os

# 1. Setup
# Look in the root folder '.' instead of 'my_photos'
source_dir = '.'
family_dir = 'family_dataset'
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')


# 2. Process Files
found_count = 0
for filename in os.listdir(source_dir):
    # Only look for your specific photos (starting with 'me_')
    if filename.lower().startswith("me_") and filename.lower().endswith((".jpg", ".png", ".jpeg")):

        # Identify the emotion from the filename
        target_emo = None
        for emo in ['happy', 'neutral', 'sad', 'stressed']:
            if emo in filename.lower():
                target_emo = emo
                break

        if not target_emo:
            continue

        # Read and Detect
        img_path = os.path.join(source_dir, filename)
        img = cv2.imread(img_path)
        if img is None: continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.1, 5)

        if len(faces) == 0:
            print(f" No face detected in {filename}. Skipping.")
            continue

        for (x, y, w, h) in faces:
            # Add margin for better context
            offset = int(w * 0.15)
            y1, y2 = max(0, y-offset), min(img.shape[0], y+h+offset)
            x1, x2 = max(0, x-offset), min(img.shape[1], x+w+offset)

            crop = img[y1:y2, x1:x2]

            # Save to the family_dataset folder
            save_path = f"{family_dir}/{target_emo}/selfie_{filename}"
            cv2.imwrite(save_path, crop)
            print(f" Added {filename} to family_dataset/{target_emo}")
            found_count += 1
            break

if found_count == 0:
    print(" No photos starting with 'me_' were found. Make sure they are visible in the sidebar!")
else:
    print(f"\n SUCCESS!")

In [ ]:
import shutil

master_dir = 'MASTER_DATASET'
emotions = ['happy', 'neutral', 'sad', 'stressed']

# 1. Create Master Folders
for emo in emotions:
    os.makedirs(f"{master_dir}/{emo}", exist_ok=True)

# 2. Merge FER2013
fer_base = 'emotions/fer2013/fer2013/train'
fer_map = {'happy':'happy', 'neutral':'neutral', 'sad':'sad', 'fear':'stressed'}
for src_emo, dst_emo in fer_map.items():
    src = f"{fer_base}/{src_emo}"
    if os.path.exists(src):
        for img in os.listdir(src):
            shutil.copy(os.path.join(src, img), f"{master_dir}/{dst_emo}/fer_{img}")

# 3. Merge CK+
ck_map = {'happy':'happy', 'contempt':'neutral', 'sadness':'sad', 'fear':'stressed'}
for src_emo, dst_emo in ck_map.items():
    src = f"ck_temp/{src_emo}" # Check if 'ck_temp' exists, otherwise adjust name
    if os.path.exists(src):
        for img in os.listdir(src):
            shutil.copy(os.path.join(src, img), f"{master_dir}/{dst_emo}/ck_{img}")

# 4. Merge the Family Dataset (Videos + Selfies)
for emo in emotions:
    src = f"family_dataset/{emo}"
    if os.path.exists(src):
        for img in os.listdir(src):
            shutil.copy(os.path.join(src, img), f"{master_dir}/{emo}/{img}")

print(f" MASTER DATASET READY: {sum([len(os.listdir(f'{master_dir}/{e}')) for e in emotions])} images.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Create a temporary generator just for viewing
view_datagen = ImageDataGenerator(rescale=1./255)
view_gen = view_datagen.flow_from_directory(
    'MASTER_DATASET',
    target_size=(96, 96),
    batch_size=64, # Bigger batch so we see more variety
    class_mode='categorical',
    shuffle=True # Mix them up!
)

# 2. Grab a batch of images
x_batch, y_batch = next(view_gen)

# 3. Display 4 images
plt.figure(figsize=(20, 10))
for i in range(8): # Let's show 8 images for even more proof
    plt.subplot(2, 4, i+1)
    plt.imshow(x_batch[i])

    # Get the emotion name
    label_idx = np.argmax(y_batch[i])
    label_name = list(view_gen.class_indices.keys())[label_idx]

    plt.title(f"Dataset Item: {label_name}")
    plt.axis('off')
plt.show()

# 4. Print counts of files for final peace of mind
print("\n--- FILE COUNTS PER FOLDER ---")
for emo in ['happy', 'neutral', 'sad', 'stressed']:
    count = len(os.listdir(f'MASTER_DATASET/{emo}'))
    print(f"Folder '{emo}': {count} total images (FER + CK+ + Family)")

In [ ]:
!pip install tf2onnx
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
import tf2onnx
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Advanced Preprocessing: CLAHE
# This matches your ai_engine.py logic, making the AI expert at reading dark/low-contrast images
def apply_clahe(img):
    img = (img * 255).astype(np.uint8)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(gray)
    res = cv2.cvtColor(cl, cv2.COLOR_GRAY2RGB)
    return res.astype(np.float32) / 255.0

# 2. Configuration
IMG_SIZE = (96, 96)
BATCH_SIZE = 64
master_dir = 'MASTER_DATASET'

# 3. High-Performance Generators (80/20 Split)
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    preprocessing_function=apply_clahe, # Teach AI to read enhanced images
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    master_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', color_mode='rgb', subset='training'
)

val_gen = datagen.flow_from_directory(
    master_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', color_mode='rgb', subset='validation'
)

# 4. Build A* Model (MobileNetV2 Fine-Tuning)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(96,96,3))
base_model.trainable = True
# Freeze first 100 layers to keep edge-detectors stable, unfreeze the rest for specific emotion learning
for layer in base_model.layers[:100]: layer.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')
])

# 5. Adaptive Learning Rate Schedulers
lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-7, verbose=1)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# 6. START TRAINING
print(f" Starting A* Final Training on {train_gen.samples} images...")
history = model.fit(train_gen, validation_data=val_gen, epochs=40, callbacks=[lr_reducer, early_stop])

# 7. SAVE & EXPORT
print("\n Exporting final model to ONNX...")
model.save("final.h5")
!pip install -q tf2onnx
!python -m tf2onnx.convert --keras a_star_final.h5 --output emotion_model_final.onnx --opset 13

# 8. VISUALIZE ACCURACY (Screenshot this for your Report!)
plt.figure(figsize=(10, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Figure 4.X: Final Model Training Metrics (96x96 Resolution)')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.show()

print("\n DONE! Download 'emotion_model_final.onnx' from the sidebar.")
print(f"Final Label Order: {list(train_gen.class_indices.keys())}")

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# 1. Image Preprocessing Implementation
def apply_clahe(img):
    """
    Applies Contrast Limited Adaptive Histogram Equalization (CLAHE).
    Ensures training data matches the real-time enhancement used during inference.
    """
    img = (img * 255).astype(np.uint8)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(gray)
    res = cv2.cvtColor(cl, cv2.COLOR_GRAY2RGB)
    return res.astype(np.float32) / 255.0

# 2. Configuration and Hyperparameters
IMG_SIZE = (96, 96)
BATCH_SIZE = 64
DATASET_PATH = 'MASTER_DATASET'

# 3. Data Augmentation and Generator Configuration
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    preprocessing_function=apply_clahe,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

validation_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# 4. Model Architecture Design
# Utilizing MobileNetV2 as the backbone for efficient edge deployment
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(96, 96, 3))

# Define the custom classification head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')
])

# 5. Training Callbacks
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True
)

lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-7
)

# --- STAGE 1: Head Warm-up (Feature Extraction) ---
# Base model weights are frozen to allow the custom head to converge initially
base_model.trainable = False
print("Executing Stage 1: Feature Extraction...")
model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
history_stage1 = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=10
)

# --- STAGE 2: Global Fine-Tuning ---
# Unfreezing the backbone to optimize features for facial expression recognition
base_model.trainable = True
print("Executing Stage 2: Global Fine-Tuning...")
# A significantly lower learning rate is used to preserve pre-trained knowledge
model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
history_stage2 = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=30,
    callbacks=[early_stop, lr_reducer]
)

# 6. Model Serialization and ONNX Conversion
print("Exporting model to ONNX format...")
model.save("final_emotion_recognition_model.h5")
!pip install -q tf2onnx
!python -m tf2onnx.convert --keras final_emotion_recognition_model.h5 --output emotion_model_final.onnx --opset 13

# 7. Performance Visualization
accuracy = history_stage1.history['accuracy'] + history_stage2.history['accuracy']
val_accuracy = history_stage1.history['val_accuracy'] + history_stage2.history['val_accuracy']

plt.figure(figsize=(10, 6))
plt.plot(accuracy, label='Training Accuracy')
plt.plot(val_accuracy, label='Validation Accuracy')
plt.axvline(x=9, color='red', linestyle='--', label='Unfreeze Point')
plt.title('Training and Validation Accuracy (96x96 Resolution)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

print(f"Training Complete. Target Labels: {list(train_generator.class_indices.keys())}")

In [ ]:
import os, shutil, zipfile, cv2, numpy as np

# 1. Download and Extract FER2013 Dataset
print("Downloading FER2013 dataset...")
!wget -q https://github.com/mh-meraj/FER2013-Dataset/archive/refs/heads/main.zip -O fer_data.zip
!unzip -q fer_data.zip

# 2. Extract CK+ Dataset
print("Extracting CK+ dataset...")
with zipfile.ZipFile("ck_data.zip", 'r') as z:
    z.extractall("ck_temp")

# 3. Create Master Directory Structure
MASTER_DIR = 'MASTER_DATASET'
EMOTIONS = ['happy', 'neutral', 'sad', 'stressed']
for emo in EMOTIONS:
    os.makedirs(f"{MASTER_DIR}/{emo}", exist_ok=True)

# 4. Integrate FER2013 (Mapping 'fear' to 'stressed')
fer_base = 'FER2013-Dataset-main/train'
fer_map = {'happy':'happy', 'neutral':'neutral', 'sad':'sad', 'fear':'stressed'}
for src, dst in fer_map.items():
    src_p = f"{fer_base}/{src}"
    if os.path.exists(src_p):
        for img in os.listdir(src_p):
            shutil.copy(os.path.join(src_p, img), f"{MASTER_DIR}/{dst}/fer_{img}")

# 5. Integrate CK+ (Searching for relevant subfolders)
ck_map = {'happy':'happy', 'contempt':'neutral', 'sadness':'sad', 'fear':'stressed'}
ck_search_root = 'ck_temp'
for root, dirs, files in os.walk('ck_temp'):
    if 'happy' in dirs:
        ck_search_root = root
        break

for src, dst in ck_map.items():
    src_p = f"{ck_search_root}/{src}"
    if os.path.exists(src_p):
        for img in os.listdir(src_p):
            shutil.copy(os.path.join(src_p, img), f"{MASTER_DIR}/{dst}/ck_{img}")

# 6. Extract frames from family videos (me, dad, bro)
# Logic: Take 100 frames per video, ensuring variety by skipping every 3 frames
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
video_files = [f for f in os.listdir('.') if f.endswith('.mp4')]

print("Processing family videos...")
for vid_name in video_files:
    target_emo = next((e for e in EMOTIONS if e in vid_name.lower()), None)
    if target_emo:
        cap = cv2.VideoCapture(vid_name)
        saved_count = 0
        frame_idx = 0
        while cap.isOpened() and saved_count < 100:
            ret, frame = cap.read()
            if not ret: break
            frame_idx += 1
            if frame_idx % 3 == 0:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                faces = face_cascade.detectMultiScale(gray, 1.3, 5)
                for (x, y, w, h) in faces:
                    crop = frame[y:y+h, x:x+w]
                    cv2.imwrite(f"{MASTER_DIR}/{target_emo}/vid_{vid_name}_{saved_count}.jpg", crop)
                    saved_count += 1
                    break
        cap.release()

# 7. Integrate individual selfies
print("Processing selfies...")
for filename in os.listdir('.'):
    if filename.lower().startswith("me_") and filename.lower().endswith((".jpg", ".png", ".jpeg")):
        target_emo = next((e for e in EMOTIONS if e in filename.lower()), None)
        if target_emo:
            img = cv2.imread(filename)
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.1, 5)
            for (x, y, w, h) in faces:
                crop = img[y:y+h, x:x+w]
                cv2.imwrite(f"{MASTER_DIR}/{target_emo}/selfie_{filename}", crop)
                break

total_images = sum([len(os.listdir(f"{MASTER_DIR}/{e}")) for e in EMOTIONS])
print(f"Data engineering complete. Total images in MASTER_DATASET: {total_images}")

In [ ]:
import os
import shutil
import zipfile
import cv2
import numpy as np

# 1. Directory Initialization
MASTER_DIR = 'MASTER_DATASET'
EMOTIONS = ['happy', 'neutral', 'sad', 'stressed']
for emo in EMOTIONS:
    if os.path.exists(os.path.join(MASTER_DIR, emo)):
        shutil.rmtree(os.path.join(MASTER_DIR, emo))
    os.makedirs(os.path.join(MASTER_DIR, emo), exist_ok=True)

# 2. CK+ Dataset Extraction
print("Extracting CK+ dataset archive...")
if os.path.exists("ck_data.zip"):
    with zipfile.ZipFile("ck_data.zip", 'r') as zip_ref:
        zip_ref.extractall("ck_temp")
else:
    print("Error: ck_data.zip not found in current directory.")

# 3. FER2013 Data Integration
# Mapping 'fear' images to 'stressed' class to align with project scope
fer_base_path = 'emotions/fer2013/fer2013/train'
fer_mapping = {'happy':'happy', 'neutral':'neutral', 'sad':'sad', 'fear':'stressed'}

print("Integrating FER2013 images...")
for src_class, dst_class in fer_mapping.items():
    source_path = os.path.join(fer_base_path, src_class)
    if os.path.exists(source_path):
        for img_name in os.listdir(source_path):
            shutil.copy(os.path.join(source_path, img_name),
                        os.path.join(MASTER_DIR, dst_class, f"fer_{img_name}"))

# 4. CK+ Data Integration
ck_mapping = {'happy':'happy', 'contempt':'neutral', 'sadness':'sad', 'fear':'stressed'}
ck_root = 'ck_temp'
# Automated search for target subfolders within the extracted zip
for root, dirs, files in os.walk('ck_temp'):
    if 'happy' in dirs:
        ck_root = root
        break

print("Integrating CK+ images...")
for src_class, dst_class in ck_mapping.items():
    source_path = os.path.join(ck_root, src_class)
    if os.path.exists(source_path):
        for img_name in os.listdir(source_path):
            shutil.copy(os.path.join(source_path, img_name),
                        os.path.join(MASTER_DIR, dst_class, f"ck_{img_name}"))

# 5. Video Frame Extraction (Family Datasets)
# Logic: Captures 100 faces per video, skipping every 3 frames for feature variety
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
video_files = [f for f in os.listdir('.') if f.endswith('.mp4')]

print("Processing family video files...")
for video_name in video_files:
    label = next((e for e in EMOTIONS if e in video_name.lower()), None)
    if label:
        cap = cv2.VideoCapture(video_name)
        count, frame_idx = 0, 0
        while cap.isOpened() and count < 100:
            ret, frame = cap.read()
            if not ret: break
            frame_idx += 1
            if frame_idx % 3 == 0:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                detected_faces = face_cascade.detectMultiScale(gray, 1.3, 5)
                for (x, y, w, h) in detected_faces:
                    face_crop = frame[y:y+h, x:x+w]
                    cv2.imwrite(os.path.join(MASTER_DIR, label, f"vid_{video_name}_{count}.jpg"), face_crop)
                    count += 1
                    break
        cap.release()
        print(f"Processed {video_name}: Extracted {count} frames.")

# 6. Local Selfie Integration
print("Processing selfie images...")
image_files = [f for f in os.listdir('.') if f.lower().endswith(('.jpg', '.jpeg'))]
for img_name in image_files:
    label = next((e for e in EMOTIONS if e in img_name.lower()), None)
    if label:
        img_raw = cv2.imread(img_name)
        gray_img = cv2.cvtColor(img_raw, cv2.COLOR_BGR2GRAY)
        faces_detected = face_cascade.detectMultiScale(gray_img, 1.1, 5)
        for (x, y, w, h) in faces_detected:
            final_crop = img_raw[y:y+h, x:x+w]
            cv2.imwrite(os.path.join(MASTER_DIR, label, f"selfie_{img_name}"), final_crop)
            break

total_count = sum([len(os.listdir(os.path.join(MASTER_DIR, e))) for e in EMOTIONS])
print(f"Dataset engineering finalized. MASTER_DATASET contains {total_count} images.")

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.preprocessing.image import load_img

def audit_dataset(directory):
    emotions = ['happy', 'neutral', 'sad', 'stressed']
    stats = {emo: {'fer': 0, 'ck': 0, 'video': 0, 'selfie': 0} for emo in emotions}

    for emo in emotions:
        path = os.path.join(directory, emo)
        if not os.path.exists(path): continue
        for f in os.listdir(path):
            if f.startswith('fer_'): stats[emo]['fer'] += 1
            elif f.startswith('ck_'): stats[emo]['ck'] += 1
            elif f.startswith('vid_'): stats[emo]['video'] += 1
            elif f.startswith('selfie_'): stats[emo]['selfie'] += 1

    print(f"{'Emotion':<10} | {'FER2013':<8} | {'CK+':<6} | {'Video':<6} | {'Selfie':<6}")
    print("-" * 50)
    for emo, data in stats.items():
        print(f"{emo:<10} | {data['fer']:<8} | {data['ck']:<6} | {data['video']:<6} | {data['selfie']:<6}")

audit_dataset('MASTER_DATASET')

# Visual Sampling for Manual Verification
def visualize_samples(directory):
    emotions = ['happy', 'neutral', 'sad', 'stressed']
    plt.figure(figsize=(12, 8))
    for i, emo in enumerate(emotions):
        path = os.path.join(directory, emo)
        files = os.listdir(path)
        img_file = files[np.random.randint(0, len(files))]
        img = load_img(os.path.join(path, img_file), target_size=(96, 96))
        plt.subplot(1, 4, i+1)
        plt.imshow(img)
        plt.title(f"{emo}\nSrc: {img_file.split('_')[0]}")
        plt.axis('off')
    plt.show()

visualize_samples('MASTER_DATASET')

In [ ]:
# Installation of Interoperability Tools
!pip install -q tf2onnx

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import cv2

# 1. Implementation of CLAHE Preprocessing
def apply_clahe_enhancement(img):
    img_uint8 = (img * 255).astype(np.uint8)
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    final_rgb = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)
    return final_rgb.astype(np.float32) / 255.0

# 2. Hyperparameters and Generator Configuration
IMG_DIMS = (96, 96)
BATCH_SIZE = 64

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
    preprocessing_function=apply_clahe_enhancement,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    'MASTER_DATASET', target_size=IMG_DIMS, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training'
)

val_generator = datagen.flow_from_directory(
    'MASTER_DATASET', target_size=IMG_DIMS, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation'
)

# 3. Architecture Construction
backbone = MobileNetV2(weights='imagenet', include_top=False, input_shape=(96, 96, 3))
backbone.trainable = False

model = models.Sequential([
    backbone,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.BatchNormalization(),
    layers.Dropout(0.6),
    layers.Dense(4, activation='softmax')
])

# 4. Phase 1: Classification Head Alignment
model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
print("Starting Phase 1: Classification Head Training...")
model.fit(train_generator, validation_data=val_generator, epochs=10)

# 5. Phase 2: Selective Backbone Fine-Tuning
# Unfreeze more of the backbone for higher adaptability
backbone.trainable = True
for layer in backbone.layers[:-60]:
    layer.trainable = False

# Use a slightly more aggressive learning rate
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

print("Starting Optimized Fine-Tuning...")
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=40,
    callbacks=[early_stop, lr_reducer]
)

# 6. Serialization and ONNX Export
model.save("emotion_recognition_model.h5")
print("Model serialized. Initiating conversion to ONNX format...")
import tf2onnx
spec = (tf.TensorSpec((None, 96, 96, 3), tf.float32, name="input"),)
tf2onnx.convert.from_keras(model, input_signature=spec, opset=13, output_path="emotion_model_final.onnx")

# 7. Metrics Visualization
plt.figure(figsize=(10, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Performance Metrics: Multi-Dataset Emotion Recognition')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

print(f"Deployment labels: {list(train_generator.class_indices.keys())}")

In [ ]:
# --- PHASE 2: OPTIMIZED FINE-TUNING ---

# 1. Callback Definitions
# Early Stopping prevents overfitting by capturing the weights at the point of minimum validation loss
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True
)

# ReduceLROnPlateau prevents the model from oscillating around local minima by decaying the learning rate
lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-7
)

# 2. Backbone Re-configuration
# Unfreezing the final 60 layers to allow for subject-specific feature optimization
backbone.trainable = True
for layer in backbone.layers[:-60]:
    layer.trainable = False

# 3. Model Compilation
# Using a learning rate of 1e-4 for stable fine-tuning gradients
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Backbone reconfiguration complete. Commencing Phase 2 training...")

# 4. Execution of Fine-Tuning
# Ensure 'val_generator' matches the variable name used in your data loading cell
history_fine_tuning = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=40,
    callbacks=[early_stop, lr_reducer]
)

# 5. Serialization and Export Implementation
print("Phase 2 complete. Serializing model for deployment...")
model.save("final_biometric_model.h5")

!pip install -q tf2onnx
!python -m tf2onnx.convert --keras final_biometric_model.h5 --output emotion_model_final.onnx --opset 13

print("Success. The optimized model 'emotion_model_final.onnx' is ready for download.")

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, regularizers
import matplotlib.pyplot as plt
import numpy as np
import cv2

# 1. Standardized Preprocessing
# Utilizing the native MobileNetV2 scaling to ensure compatibility with pre-trained weights
def preprocess_standard(img):
    return tf.keras.applications.mobilenet_v2.preprocess_input(img)

# 2. Configuration
IMG_SIZE = (96, 96)
BATCH_SIZE = 32 # Reduced batch size for finer gradient updates
DATASET_PATH = 'MASTER_DATASET'

# 3. Robust Data Generators
# Increased horizontal flip and rotation to force feature generalization
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    preprocessing_function=preprocess_standard,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    DATASET_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training'
)

validation_generator = datagen.flow_from_directory(
    DATASET_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', shuffle=False
)

# 4. Model Construction
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(96, 96, 3))

# Global Unfreeze: Allowing all layers to adapt to the facial expression domain
base_model.trainable = True

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')
])

# 5. Optimization Strategy
# Using a very low learning rate (1e-5) to prevent weight explosion
# Using SGD with Momentum for superior convergence stability over Adam
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-5, momentum=0.9)

model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Callbacks
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=5)

# 6. Training Execution
print("Commencing Global Fine-Tuning Strategy...")
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=50,
    callbacks=[early_stop, lr_reducer]
)

# 7. Final Model Evaluation
print("Generating Final Classification Metrics...")
val_loss, val_acc = model.evaluate(validation_generator)
print(f"Final Validation Accuracy: {val_acc:.4f}")

# 8. Export to ONNX via CLI
model.save("global_optimized_model.h5")
!pip install -q tf2onnx
!python -m tf2onnx.convert --keras global_optimized_model.h5 --output emotion_model_final.onnx --opset 13

# 9. Performance Visualization
plt.figure(figsize=(10, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Emotion Recognition: Global Optimization Performance')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 1. Save the model to a standard H5 file
model.save("temp_model.h5")

# 2. Use the CLI converter (this bypasses the KeyError)
print("Converting to ONNX via Command Line...")
!python -m tf2onnx.convert --keras temp_model.h5 --output emotion_model_final.onnx --opset 13

print("Success! Download 'emotion_model_final.onnx' from the sidebar.")


In [3]:
# ==========================================
# 1. DEPENDENCIES & ENVIRONMENT SETUP
# ==========================================
!pip install -q tf2onnx onnxruntime seaborn

import os
import shutil
import zipfile
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam, SGD
from sklearn.metrics import confusion_matrix, classification_report
import tf2onnx

# ==========================================
# 2. DATA ENGINEERING & FUSION
# ==========================================
MASTER_DIR = 'MASTER_DATASET'
EMOTIONS = ['happy', 'neutral', 'sad', 'stressed']

# Initialize Directories
for emo in EMOTIONS:
    os.makedirs(os.path.join(MASTER_DIR, emo), exist_ok=True)

# A. Download and Extract FER2013
print("Downloading FER2013...")
!wget -q https://github.com/mh-meraj/FER2013-Dataset/archive/refs/heads/main.zip -O fer_data.zip
!unzip -q fer_data.zip
fer_base = 'FER2013-Dataset-main/train'
fer_map = {'happy':'happy', 'neutral':'neutral', 'sad':'sad', 'fear':'stressed'}
for src, dst in fer_map.items():
    src_p = f"{fer_base}/{src}"
    if os.path.exists(src_p):
        for img in os.listdir(src_p):
            shutil.copy(os.path.join(src_p, img), os.path.join(MASTER_DIR, dst, f"fer_{img}"))

# B. Extract and Integrate CK+
if os.path.exists("ck_data.zip"):
    print("Integrating CK+ dataset...")
    with zipfile.ZipFile("ck_data.zip", 'r') as z:
        z.extractall("ck_temp")
    ck_map = {'happy':'happy', 'contempt':'neutral', 'sadness':'sad', 'fear':'stressed'}
    for root, dirs, files in os.walk('ck_temp'):
        if 'happy' in dirs:
            for src, dst in ck_map.items():
                src_p = os.path.join(root, src)
                if os.path.exists(src_p):
                    for img in os.listdir(src_p):
                        shutil.copy(os.path.join(src_p, img), os.path.join(MASTER_DIR, dst, f"ck_{img}"))

# C. Process Family Videos (MP4)
print("Processing family video frames...")
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
for vid in [f for f in os.listdir('.') if f.endswith('.mp4')]:
    target_emo = next((e for e in EMOTIONS if e in vid.lower()), None)
    if target_emo:
        cap = cv2.VideoCapture(vid)
        count, frame_idx = 0, 0
        while cap.isOpened() and count < 100:
            ret, frame = cap.read()
            if not ret: break
            frame_idx += 1
            if frame_idx % 3 == 0:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                faces = face_cascade.detectMultiScale(gray, 1.3, 5)
                for (x, y, w, h) in faces:
                    cv2.imwrite(os.path.join(MASTER_DIR, target_emo, f"vid_{vid}_{count}.jpg"), frame[y:y+h, x:x+w])
                    count += 1
                    break
        cap.release()

# D. Integrate Selfies (JPG)
print("Integrating selfies...")
for img_name in [f for f in os.listdir('.') if f.lower().endswith(('.jpg', '.jpeg'))]:
    target_emo = next((e for e in EMOTIONS if e in img_name.lower()), None)
    if target_emo:
        img_raw = cv2.imread(img_name)
        gray = cv2.cvtColor(img_raw, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.1, 5)
        for (x, y, w, h) in faces:
            cv2.imwrite(os.path.join(MASTER_DIR, target_emo, f"selfie_{img_name}"), img_raw[y:y+h, x:x+w])
            break

print(f"Dataset Fused. Total Images: {sum([len(os.listdir(os.path.join(MASTER_DIR, e))) for e in EMOTIONS])}")

# ==========================================
# 3. TRAINING CONFIGURATION
# ==========================================
IMG_SIZE = (96, 96)
BATCH_SIZE = 64

datagen = ImageDataGenerator(
    rotation_range=20, width_shift_range=0.15, height_shift_range=0.15,
    horizontal_flip=True, brightness_range=[0.8, 1.2],
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(MASTER_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', subset='training', classes=EMOTIONS)
val_gen = datagen.flow_from_directory(MASTER_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', subset='validation', classes=EMOTIONS)

# ==========================================
# 4. MODEL ARCHITECTURE & PHASE 1
# ==========================================
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(96, 96, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.BatchNormalization(),
    layers.Dropout(0.6),
    layers.Dense(4, activation='softmax')
])

print("Phase 1: Classification Head Stabilization...")
model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
history1 = model.fit(train_gen, validation_data=val_gen, epochs=10)

# ==========================================
# 5. PHASE 2: GLOBAL FINE-TUNING
# ==========================================
base_model.trainable = True
for layer in base_model.layers[:-50]: layer.trainable = False

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True)
lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.2, patience=4)

# Using SGD for superior fine-tuning stability
model.compile(optimizer=SGD(learning_rate=1e-4, momentum=0.9), loss='categorical_crossentropy', metrics=['accuracy'])

print("Phase 2: Global Fine-Tuning...")
history2 = model.fit(train_gen, validation_data=val_gen, epochs=40, callbacks=[early_stop, lr_reducer])

# ==========================================
# 6. EVALUATION & VISUALIZATION
# ==========================================
# Generate Optimization Graph
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
plt.figure(figsize=(10, 6))
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.axvline(x=9, color='red', linestyle='--', label='Unfreeze Point')
plt.title('Optimization Performance Graph')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Generate Confusion Matrix
print("Generating Confusion Matrix...")
test_datagen = ImageDataGenerator(preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input, validation_split=0.2)
test_gen = test_datagen.flow_from_directory(MASTER_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', subset='validation', shuffle=False, classes=EMOTIONS)

Y_pred = model.predict(test_gen)
y_pred = np.argmax(Y_pred, axis=1)
cm = confusion_matrix(test_gen.classes, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=EMOTIONS, yticklabels=EMOTIONS)
plt.title('Final Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

print("\nClassification Report:")
print(classification_report(test_gen.classes, y_pred, target_names=EMOTIONS))

# ==========================================
# 7. ONNX EXPORT
# ==========================================
model.save("final_emotion_model.h5")
print("Converting to ONNX...")
!python -m tf2onnx.convert --keras final_emotion_model.h5 --output emotion_model_final.onnx --opset 13
print("Success. Download emotion_model_final.onnx from the sidebar.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 105.7 MB/s eta 0:00:00
[fer_data.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of fer_data.zip or
        fer_data.zip.zip, and cannot find fer_data.zip.ZIP, period.
Integrating CK+ dataset...
Processing family video frames...


KeyboardInterrupt: 

In [6]:
import os
import shutil
import zipfile
import cv2
import numpy as np

# 1. Configuration of Directory Paths
MASTER_DIR = 'MASTER_DATASET'
EMOTIONS = ['happy', 'neutral', 'sad', 'stressed']
# Localizing FER2013 based on your specific directory structure
FER_BASE = 'emotions/fer2013/fer2013/train'

# Initialize master directories
for emo in EMOTIONS:
    if os.path.exists(os.path.join(MASTER_DIR, emo)):
        shutil.rmtree(os.path.join(MASTER_DIR, emo))
    os.makedirs(os.path.join(MASTER_DIR, emo), exist_ok=True)

# 2. CK+ Dataset Extraction
if os.path.exists("ck_data.zip"):
    with zipfile.ZipFile("ck_data.zip", 'r') as z:
        z.extractall("ck_temp")

# 3. Data Fusion Logic
# Mapping source categories to project-specific target classes
fer_mapping = {'happy':'happy', 'neutral':'neutral', 'sad':'sad', 'fear':'stressed'}
ck_mapping = {'happy':'happy', 'contempt':'neutral', 'sadness':'sad', 'fear':'stressed'}

print("Consolidating FER2013 images...")
for src_class, dst_class in fer_mapping.items():
    source_p = os.path.join(FER_BASE, src_class)
    if os.path.exists(source_p):
        for img in os.listdir(source_p):
            shutil.copy(os.path.join(source_p, img), os.path.join(MASTER_DIR, dst_class, f"fer_{img}"))

print("Consolidating CK+ images...")
ck_root = 'ck_temp'
for root, dirs, files in os.walk('ck_temp'):
    if 'happy' in dirs:
        ck_root = root
        break
for src_class, dst_class in ck_mapping.items():
    source_p = os.path.join(ck_root, src_class)
    if os.path.exists(source_p):
        for img in os.listdir(source_p):
            shutil.copy(os.path.join(source_p, img), os.path.join(MASTER_DIR, dst_class, f"ck_{img}"))

# 4. Automated Video Frame Extraction (MP4)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
print("Extracting frames from video datasets...")
for vid_file in [f for f in os.listdir('.') if f.endswith('.mp4')]:
    label = next((e for e in EMOTIONS if e in vid_file.lower()), None)
    if label:
        cap = cv2.VideoCapture(vid_file)
        count, frame_idx = 0, 0
        while cap.isOpened() and count < 100:
            ret, frame = cap.read()
            if not ret: break
            frame_idx += 1
            if frame_idx % 3 == 0:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                faces = face_cascade.detectMultiScale(gray, 1.3, 5)
                for (x, y, w, h) in faces:
                    cv2.imwrite(os.path.join(MASTER_DIR, label, f"vid_{vid_file}_{count}.jpg"), frame[y:y+h, x:x+w])
                    count += 1
                    break
        cap.release()

# 5. Selfie Integration (JPG/JPEG)
print("Integrating individual selfie images...")
for img_file in [f for f in os.listdir('.') if f.lower().endswith(('.jpg', '.jpeg'))]:
    label = next((e for e in EMOTIONS if e in img_file.lower()), None)
    if label:
        img_raw = cv2.imread(img_file)
        if img_raw is not None:
            gray = cv2.cvtColor(img_raw, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.1, 5)
            for (x, y, w, h) in faces:
                cv2.imwrite(os.path.join(MASTER_DIR, label, f"selfie_{img_file}"), img_raw[y:y+h, x:x+w])
                break

image_count = sum([len(os.listdir(os.path.join(MASTER_DIR, e))) for e in EMOTIONS])
print(f"Data Engineering complete. Master directory contains {image_count} images.")

Consolidating FER2013 images...
Consolidating CK+ images...
Extracting frames from video datasets...
Integrating individual selfie images...
Data Engineering complete. Master directory contains 22628 images.


In [ ]:
# 1. Environmental Setup
!pip install -q tf2onnx onnxruntime seaborn

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam, SGD
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import tf2onnx
import numpy as np

# 2. Hyperparameters
IMG_SIZE = (96, 96)
BATCH_SIZE = 64
DATA_PATH = 'MASTER_DATASET'

# 3. Data Generator Implementation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(DATA_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', subset='training')
validation_generator = datagen.flow_from_directory(DATA_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', subset='validation', shuffle=False)

# 4. Architecture Design
backbone = MobileNetV2(weights='imagenet', include_top=False, input_shape=(96, 96, 3))
backbone.trainable = False

model = models.Sequential([
    backbone,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    layers.BatchNormalization(),
    layers.Dropout(0.6),
    layers.Dense(4, activation='softmax')
])

# 5. Training Phase 1: Classification Head Alignment
model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
print("Executing Phase 1: Classification Head Stabilization...")
history1 = model.fit(train_generator, validation_data=validation_generator, epochs=10)

# 6. Training Phase 2: Global Fine-Tuning
backbone.trainable = True
for layer in backbone.layers[:-40]:
    layer.trainable = False

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True)
lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.2, patience=4)

model.compile(optimizer=SGD(learning_rate=1e-4, momentum=0.9), loss='categorical_crossentropy', metrics=['accuracy'])
print("Executing Phase 2: backbone Fine-Tuning...")
history2 = model.fit(train_generator, validation_data=validation_generator, epochs=40, callbacks=[early_stop, lr_reducer])

# 7. Visualization of Training Performance
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
plt.figure(figsize=(10, 6))
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.axvline(x=9, color='red', linestyle='--', label='Unfreeze Point')
plt.title('Emotion Recognition Optimization Graph')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# 8. Final Evaluation and Matrix Generation
print("Generating Final Evaluation Metrics...")
Y_pred = model.predict(validation_generator)
y_pred = np.argmax(Y_pred, axis=1)
cm = confusion_matrix(validation_generator.classes, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=EMOTIONS, yticklabels=EMOTIONS)
plt.title('Final Confusion Matrix: Multi-Dataset Model')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

print("\nClassification Report:")
print(classification_report(validation_generator.classes, y_pred, target_names=EMOTIONS))

# 9. Model Serialization and ONNX Export
model.save("optimized_emotion_model.h5")
spec = (tf.TensorSpec((None, 96, 96, 3), tf.float32, name="input"),)
tf2onnx.convert.from_keras(model, input_signature=spec, opset=13, output_path="emotion_model_final.onnx")

print(f"Workflow complete. Final Deployment Labels: {list(train_generator.class_indices.keys())}")

Found 18103 images belonging to 4 classes.
Found 4525 images belonging to 4 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Executing Phase 1: Classification Head Stabilization...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
283/283 ━━━━━━━━━━━━━━━━━━━━ 128s 385ms/step - accuracy: 0.3948 - loss: 6.6244 - val_accuracy: 0.4884 - val_loss: 2.6726
Epoch 2/10
283/283 ━━━━━━━━━━━━━━━━━━━━ 83s 293ms/step - accuracy: 0.4806 - loss: 2.3463 - val_accuracy: 0.4480 - val_loss: 1.7793
Epoch 3/10
120/283 ━━━━━━━━━━━━━━━━━━━━ 39s 241ms/step - accuracy: 0.4839 - loss: 1.6178